## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import sys

sys.setrecursionlimit(300000)

def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    idx = 0
    m = data[idx]
    idx += 1

    s = data[idx]
    idx += 1

    t = data[idx]
    idx += 1

    a = data[idx:idx + m]
    res = []

    z = 0

    def op0():
        res.append(0)

        for i in range(m):
            if a[i] == s:
                a[i] = t
            elif a[i] == t:
                a[i] = s

    def op2(w):
        w %= m
        if w < 0:
            w += m

        if w == 0:
            return

        res.append(w)

        for i in range(m):
            a[i] = (a[i] + w) % m

    def op1(w):
        if w == 0:
            return

        res.append(-w)

        for i in range(m):
            a[i] ^= w

    def find_pair(x, y):
        d = (y - x + m - z + m) % m

        l = 0
        r = 0
        step = m // 2

        while step >= 2 * z:
            if d >= step:
                d -= step
                r += step // 2
            else:
                l += step // 2

            step >>= 1

        l += m // 2
        l += x & (z - 1)
        r += x & (z - 1)

        return l, r

    def change_pos(x, y):
        if (x // z) % 2 == (y // z) % 2:
            if (x // z) % 2 == 0:
                mid = (x & (z - 1)) + z
            else:
                mid = x & (z - 1)

            change_pos(x, mid)
            change_pos(y, mid)
            change_pos(x, mid)
            return

        p1, _ = find_pair(s, t)
        p2, _ = find_pair(x, y)

        op2((p2 - x + m) % m)
        op1(p2 ^ p1)
        op2((s - p1 + m) % m)

        op0()

        op2((p1 - s + m) % m)
        op1(p2 ^ p1)
        op2((x - p2 + m) % m)

    def build(b, length):
        mark = [0] * length

        for x in b:
            if x >= length:
                return False, []
            mark[x] = 1

        for x in mark:
            if not x:
                return False, []

        if length == 1:
            return True, []

        left = [0] * (length // 2)
        right = [0] * (length // 2)

        for i in range(length // 2):
            left[i] = b[i * 2] // 2
            right[i] = b[i * 2 + 1] // 2

        ok1, v_left = build(left, length // 2)
        ok2, v_right = build(right, length // 2)

        if not ok1 or not ok2:
            return False, []

        cur = []

        if b[0] & 1:
            if length == 2:
                cur.append(1)
            else:
                cur.append(-1)

        q1 = 0

        for x in v_left:
            if x > 0:
                cur.append(-1)
                cur.append(1)
            else:
                cur.append(x * 2)
                q1 ^= (-x) * 2

        if q1 != 0:
            cur.append(-q1)

        q2 = 0

        for x in v_right:
            if x > 0:
                cur.append(1)
                cur.append(-1)
            else:
                cur.append(x * 2)
                q2 ^= (-x) * 2

        if (q2 & (length // 2)) != (q1 & (length // 2)):
            return False, []

        if q1 >= length // 2:
            q1 -= length // 2

        if q2 >= length // 2:
            q2 -= length // 2

        if q1 != q2:
            return False, []

        ans = []

        for x in cur:
            if not ans:
                ans.append(x)
            else:
                if x < 0 and ans[-1] < 0:
                    ans[-1] = -((-ans[-1]) ^ (-x))

                    if ans[-1] == 0:
                        ans.pop()
                else:
                    ans.append(x)

        return True, ans

    z = (s - t + m) % m
    z &= -z

    if z == 0:
        z = m

    if z > 1:
        tmp = [0] * z

        for i in range(z):
            tmp[i] = a[i] & (z - 1)

        ok, ops = build(tmp, z)

        if not ok:
            print(-1)
            return

        for x in ops:
            if x > 0:
                op2(x)
            else:
                op1(-x)

    for r in range(z):
        b = []

        for i in range(r, m, z):
            b.append(a[i])

        b.sort()

        k = 0
        ok = True

        for i in range(r, m, z):
            if b[k] != i:
                ok = False
                break
            k += 1

        if not ok:
            print(-1)
            return

        for i in range(r, m, z):
            if a[i] != i:
                change_pos(i, a[i])

    out = [str(len(res))]

    for x in res:
        if x == 0:
            out.append("0")
        elif x < 0:
            out.append(f"1 {-x}")
        else:
            out.append(f"2 {x}")

    sys.stdout.write("\n".join(out))


if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
import sys
import heapq

stations = []

def solve(n, l, maxn, s):
    st = [(v[0], v[1]) for v in stations]
    st.sort()

    while st and st[-1][0] > l:
        st.pop()

    n = len(st)

    if l <= maxn:
        return True

    inf = 1 << 60
    dist = [inf] * n

    for j in range(n):
        if st[j][0] <= maxn:
            dist[j] = min(dist[j], st[j][1])
        else:
            break

    q = []

    for j in range(n):
        if dist[j] < inf:
            heapq.heappush(q, (dist[j], j))

    while q:
        d, i = heapq.heappop(q)

        if d != dist[i]:
            continue

        if d > s:
            continue

        pi = st[i][0]

        if l - pi <= maxn:
            return True

        for j in range(i + 1, n):
            if st[j][0] - pi > maxn:
                break

            nd = d + st[j][1]

            if nd < dist[j]:
                dist[j] = nd
                heapq.heappush(q, (nd, j))

    return False


def main():
    global stations

    data = list(map(int, sys.stdin.buffer.read().split()))
    idx = 0
    ans = []

    while idx + 4 <= len(data):
        n = data[idx]
        l = data[idx + 1]
        maxn = data[idx + 2]
        s = data[idx + 3]
        idx += 4

        stations = []

        for _ in range(n):
            p = data[idx]
            c = data[idx + 1]
            idx += 2
            stations.append([p, c])

        ans.append("Yes" if solve(n, l, maxn, s) else "No")

    sys.stdout.write("\n".join(ans))


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
import sys

BASE = 911382323
MASK = (1 << 64) - 1


def manacher(s):
    n = len(s)
    m = n * 2 + 2

    t = bytearray(m + 1)
    t[0] = 1
    t[1] = 2

    k = 2
    for ch in s:
        t[k] = ch
        k += 1
        t[k] = 2
        k += 1

    t[m] = 3

    p = [0] * (m + 1)
    mx = 0
    idx = 0

    for i in range(1, m):
        if mx > i:
            v = p[2 * idx - i]
            w = mx - i
            p[i] = v if v < w else w
        else:
            p[i] = 1

        x = p[i]

        while t[i + x] == t[i - x]:
            x += 1

        p[i] = x

        if i + x > mx:
            mx = i + x
            idx = i

    return p, m


def main():
    data = sys.stdin.buffer.read().split()
    if not data:
        return

    n = int(data[0])
    s1 = data[1]
    s2 = data[2]

    pa, m = manacher(s1)
    pb, _ = manacher(s2)

    rs1 = s1[::-1]

    h1 = [0] * (n + 1)
    h2 = [0] * (n + 1)
    pw = [1] * (n + 1)

    base = BASE
    mask = MASK

    for i in range(n):
        h1[i + 1] = (h1[i] * base + rs1[i] + 7) & mask
        h2[i + 1] = (h2[i] * base + s2[i] + 7) & mask
        pw[i + 1] = (pw[i] * base) & mask

    ans = 1

    for i in range(2, m + 1):
        length = pa[i]
        if pb[i - 2] > length:
            length = pb[i - 2]

        x = i - length
        y = i - 2 + length

        extra = 0

        if 0 <= x and y < m:
            if x & 1:
                extra = 1

                j = (x - 3) // 2
                k = (y - 1) // 2

                if j >= 0 and k < n and s1[j] == s2[k]:
                    lim = j + 1
                    if n - k < lim:
                        lim = n - k

                    l = 1
                    r = lim
                    pos = n - 1 - j

                    while l < r:
                        mid = (l + r + 1) >> 1

                        v1 = (h1[pos + mid] - h1[pos] * pw[mid]) & mask
                        v2 = (h2[k + mid] - h2[k] * pw[mid]) & mask

                        if v1 == v2:
                            l = mid
                        else:
                            r = mid - 1

                    extra += l * 2

            elif x >= 2 and y >= 2:
                j = x // 2 - 1
                k = y // 2 - 1

                if j >= 0 and k < n and s1[j] == s2[k]:
                    lim = j + 1
                    if n - k < lim:
                        lim = n - k

                    l = 1
                    r = lim
                    pos = n - 1 - j

                    while l < r:
                        mid = (l + r + 1) >> 1

                        v1 = (h1[pos + mid] - h1[pos] * pw[mid]) & mask
                        v2 = (h2[k + mid] - h2[k] * pw[mid]) & mask

                        if v1 == v2:
                            l = mid
                        else:
                            r = mid - 1

                    extra = l * 2

        cur = length + extra - 1

        if cur > ans:
            ans = cur

    print(ans)


if __name__ == "__main__":
    main()

## D 优惠券

In [ ]:
import sys

LIMIT = 100000

state = [0] * (LIMIT + 1)
prev_pos = [0] * (LIMIT + 1)


class Tree:
    def __init__(self, size):
        self.size = size
        self.arr = [0] * (size + 2)

    def update(self, pos, val):
        size = self.size
        arr = self.arr

        while pos <= size:
            arr[pos] += val
            pos += pos & -pos

    def query(self, pos):
        total = 0
        arr = self.arr

        while pos > 0:
            total += arr[pos]
            pos -= pos & -pos

        return total

    def locate(self, kth):
        pos = 0
        step = 1 << (self.size.bit_length() - 1)
        arr = self.arr

        while step:
            nxt = pos + step

            if nxt <= self.size and arr[nxt] < kth:
                pos = nxt
                kth -= arr[nxt]

            step >>= 1

        return pos + 1


def run():
    data = sys.stdin.buffer.read().split()
    ptr = 0
    result = []

    while ptr < len(data):
        cnt = int(data[ptr])
        ptr += 1

        tree = Tree(cnt)
        changed = []
        answer = -1

        for idx in range(1, cnt + 1):
            typ = data[ptr]
            ptr += 1

            if typ == b'?':
                if answer == -1:
                    tree.update(idx, 1)
                continue

            num = int(data[ptr])
            ptr += 1

            if answer != -1:
                continue

            if prev_pos[num] == 0:
                changed.append(num)

            if typ == b'I':
                state[num] += 1
            else:
                state[num] -= 1

            if state[num] < 0 or state[num] > 1:
                left_count = tree.query(prev_pos[num] - 1)
                all_count = tree.query(cnt)

                if all_count == left_count:
                    answer = idx
                else:
                    place = tree.locate(left_count + 1)
                    tree.update(place, -1)

                    if state[num] < 0:
                        state[num] = 0
                    else:
                        state[num] = 1

            prev_pos[num] = idx

        result.append(str(answer))

        for num in changed:
            state[num] = 0
            prev_pos[num] = 0

    sys.stdout.write("\n".join(result))


if __name__ == "__main__":
    run()

## E 任意点

In [ ]:
import sys

class MergeSet:
    def __init__(self, size):
        self.parent = list(range(size))

    def root(self, node):
        while self.parent[node] != node:
            self.parent[node] = self.parent[self.parent[node]]
            node = self.parent[node]
        return node

    def join(self, u, v):
        ru = self.root(u)
        rv = self.root(v)
        if ru != rv:
            self.parent[ru] = rv


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    total = data[0]
    points = []
    idx = 1

    for _ in range(total):
        a = data[idx]
        b = data[idx + 1]
        idx += 2
        points.append((a, b))

    group = MergeSet(total)

    same_x = {}
    same_y = {}

    for i, (a, b) in enumerate(points):
        if a in same_x:
            group.join(i, same_x[a])
        else:
            same_x[a] = i

        if b in same_y:
            group.join(i, same_y[b])
        else:
            same_y[b] = i

    count = 0

    for i in range(total):
        if group.root(i) == i:
            count += 1

    print(count - 1)


if __name__ == "__main__":
    main()

## F 通配符匹配

In [ ]:
#include <bits/stdc++.h>
using namespace std;

using ull = unsigned long long;
static const ull BASE = 1315423911ull;

struct Block {
    int offset;
    string str;
    ull h;
};

struct Segment {
    int len = 0;
    vector<Block> blocks; 
    int anchorId = -1;
    string anchor;
    int anchorOffset = 0; 
    vector<int> lps;
};

ull getStrHash(const string &t) {
    ull h = 0;
    for (char c : t) h = h * BASE + (ull)(c - 'a' + 1);
    return h;
}

vector<int> buildLPS(const string &p) {
    vector<int> lps((int)p.size(), 0);
    for (int i = 1, j = 0; i < (int)p.size(); i++) {
        while (j > 0 && p[i] != p[j]) j = lps[j - 1];
        if (p[i] == p[j]) j++;
        lps[i] = j;
    }
    return lps;
}

vector<int> kmpSearch(const string &s, const string &p, const vector<int> &lps) {
    vector<int> occ;
    if (p.empty()) return occ;
    for (int i = 0, j = 0; i < (int)s.size(); i++) {
        while (j > 0 && s[i] != p[j]) j = lps[j - 1];
        if (s[i] == p[j]) j++;
        if (j == (int)p.size()) {
            occ.push_back(i - j + 1);
            j = lps[j - 1];
        }
    }
    return occ;
}

Segment parseSegment(const string &seg) {
    Segment res;
    res.len = (int)seg.size();

    int i = 0;
    while (i < (int)seg.size()) {
        if (seg[i] == '?') {
            i++;
            continue;
        }
        int j = i;
        while (j < (int)seg.size() && seg[j] != '?') j++;
        string t = seg.substr(i, j - i);
        res.blocks.push_back({i, t, getStrHash(t)});
        i = j;
    }

    for (int id = 0; id < (int)res.blocks.size(); id++) {
        if (res.anchorId == -1 || res.blocks[id].str.size() > res.blocks[res.anchorId].str.size()) {
            res.anchorId = id;
        }
    }

    if (res.anchorId != -1) {
        res.anchor = res.blocks[res.anchorId].str;
        res.anchorOffset = res.blocks[res.anchorId].offset;
        res.lps = buildLPS(res.anchor);
    }

    return res;
}

vector<ull> pw{1};

void ensurePow(int n) {
    while ((int)pw.size() <= n) pw.push_back(pw.back() * BASE);
}

struct RH {
    vector<ull> pref;
    RH(const string &s) {
        int n = (int)s.size();
        ensurePow(n);
        pref.assign(n + 1, 0);
        for (int i = 0; i < n; i++) {
            pref[i + 1] = pref[i] * BASE + (ull)(s[i] - 'a' + 1);
        }
    }
    ull get(int l, int r) const {
        return pref[r] - pref[l] * pw[r - l];
    }
};

bool segmentMatchAt(const Segment &seg, const string &s, const RH &rh, int st) {
    if (st < 0 || st + seg.len > (int)s.size()) return false;
    for (const auto &b : seg.blocks) {
        int l = st + b.offset;
        int r = l + (int)b.str.size();
        if (rh.get(l, r) != b.h) return false;
    }
    return true;
}

int findNextPos(const Segment &seg, const string &s, const RH &rh,
                const vector<int> &occ, int pos, int rightBound) {
    if (seg.len == 0) return pos;

    if (seg.blocks.empty()) {
        return (pos + seg.len <= rightBound) ? pos : -1;
    }

    int need = pos + seg.anchorOffset;
    auto it = lower_bound(occ.begin(), occ.end(), need);

    while (it != occ.end()) {
        int t = *it;
        int st = t - seg.anchorOffset;

        if (st + seg.len > rightBound) break;

        if (segmentMatchAt(seg, s, rh, st)) return st;
        ++it;
    }
    return -1;
}

bool matchString(const vector<Segment> &segs, bool leadStar, bool trailStar, const string &s) {
    RH rh(s);
    int m = (int)segs.size();

    if (!leadStar && !trailStar && m == 1) {
        return segs[0].len == (int)s.size() && segmentMatchAt(segs[0], s, rh, 0);
    }

    vector<vector<int>> occ(m);
    for (int i = 0; i < m; i++) {
        if (!segs[i].anchor.empty()) {
            occ[i] = kmpSearch(s, segs[i].anchor, segs[i].lps);
        }
    }

    int l = 0, r = (int)s.size();
    int L = 0, R = m - 1;

    if (!leadStar) {
        if (!segmentMatchAt(segs[0], s, rh, 0)) return false;
        l = segs[0].len;
        L = 1;
    }

    if (!trailStar) {
        int st = (int)s.size() - segs[m - 1].len;
        if (st < l) return false;
        if (!segmentMatchAt(segs[m - 1], s, rh, st)) return false;
        r = st;
        R = m - 2;
    }

    int pos = l;
    for (int i = L; i <= R; i++) {
        int st = findNextPos(segs[i], s, rh, occ[i], pos, r);
        if (st == -1) return false;
        pos = st + segs[i].len;
    }

    return true;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string pattern;
    cin >> pattern;

    string p;
    for (char c : pattern) {
        if (c == '*' && !p.empty() && p.back() == '*') continue;
        p.push_back(c);
    }

    bool leadStar = (!p.empty() && p.front() == '*');
    bool trailStar = (!p.empty() && p.back() == '*');

    vector<Segment> segs;
    string cur;
    for (char c : p) {
        if (c == '*') {
            segs.push_back(parseSegment(cur));
            cur.clear();
        } else {
            cur.push_back(c);
        }
    }
    segs.push_back(parseSegment(cur));

    int n;
    cin >> n;

    while (n--) {
        string s;
        cin >> s;
        cout << (matchString(segs, leadStar, trailStar, s) ? "YES" : "NO") << '\n';
    }

    return 0;
}

## G 汉诺塔

In [ ]:
import sys


def get_index(ch):
    return ord(ch) - ord("A")


def main():
    data = sys.stdin.read().split()
    if not data:
        return

    disk_count = int(data[0])
    moves = data[1:7]

    order = [[0] * 3 for _ in range(3)]

    for i in range(6):
        x = get_index(moves[i][0])
        y = get_index(moves[i][1])
        order[x][y] = i

    target = [0] * 3

    for x in range(3):
        choices = []

        for y in range(3):
            if y != x:
                choices.append(y)

        first = choices[0]
        second = choices[1]

        if order[x][first] < order[x][second]:
            target[x] = first
        else:
            target[x] = second

    power3 = [1] * 31

    for i in range(1, 31):
        power3[i] = power3[i - 1] * 3

    start = 0

    if target[target[start]] == start:
        answer = 2 * power3[disk_count - 1] - 1
    elif target[target[target[start]]] == start:
        answer = (1 << disk_count) - 1
    else:
        answer = power3[disk_count - 1]

    print(answer)


if __name__ == "__main__":
    main()

## H 马步距离

In [ ]:
import sys


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    x1, y1, x2, y2 = data

    a = abs(x1 - x2)
    b = abs(y1 - y2)

    if a < b:
        a, b = b, a

    if a == 1 and b == 0:
        print(3)
        return

    if a == 2 and b == 2:
        print(4)
        return

    ans = max((a + 1) // 2, (a + b + 2) // 3)

    while (ans + a + b) % 2 != 0:
        ans += 1

    print(ans)


if __name__ == "__main__":
    main()

## I 直方图最大矩形

In [ ]:
#
# 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
#
# 
# @param heights int整型一维数组 
# @return int整型
#
class Solution:
    def largestRectangleArea(self , heights: List[int]) -> int:
        n = len(heights)
        if n == 0:
            return 0

        stack = []
        ans = 0

        for i in range(n + 1):
            h = 0 if i == n else heights[i]

            while stack and heights[stack[-1]] > h:
                cur = stack.pop()
                left = -1 if not stack else stack[-1]
                width = i - left - 1
                ans = max(ans, heights[cur] * width)

            stack.append(i)

        return ans

## J 消防局的设立

In [ ]:
import sys


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    size = data[0]
    parent = [0] * (size + 1)
    depth = [0] * (size + 1)

    idx = 1
    for node in range(2, size + 1):
        parent[node] = data[idx]
        idx += 1
        depth[node] = depth[parent[node]] + 1

    order = list(range(1, size + 1))
    order.sort(key=lambda x: depth[x], reverse=True)

    placed = [0] * (size + 1)
    child_mark = [0] * (size + 1)
    grand_mark = [0] * (size + 1)

    def is_safe(node):
        p = parent[node]
        gp = parent[p] if p else 0

        if placed[node]:
            return True
        if p and placed[p]:
            return True
        if gp and placed[gp]:
            return True
        if child_mark[node] > 0:
            return True
        if grand_mark[node] > 0:
            return True
        if p and child_mark[p] > 0:
            return True

        return False

    answer = 0

    for node in order:
        if depth[node] > 2 and not is_safe(node):
            target = parent[parent[node]]

            if not placed[target]:
                placed[target] = 1
                answer += 1

                p = parent[target]
                gp = parent[p] if p else 0

                if p:
                    child_mark[p] += 1
                if gp:
                    grand_mark[gp] += 1

    for node in range(1, size + 1):
        if not is_safe(node):
            answer += 1
            break

    print(answer)


if __name__ == "__main__":
    main()